In [ ]:
spark

In [ ]:
# 1. Read the dataset from your storage volume path
df = spark.read.csv("/Volumes/students/default/filestore/Online_Retail-1.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5, truncate=False)
print(f"Total transactions: {df.count()}")

In [ ]:
# 2. Data cleaning and preprocessing transformations
from pyspark.sql.functions import col, to_timestamp, when, lit, regexp_replace

df_clean = df.filter(col("CustomerID").isNotNull())

# 1. First, reorganize the raw string format to match your professor's string requirements
# This safely converts '13/12/2010 9:09' to '12/13/2010 9:09'
df_clean = df_clean.withColumn(
    "InvoiceDate", 
    regexp_replace(col("InvoiceDate"), r"^(\d+)/(\d+)/", r"$2/$1/")
)

# 2. Now your professor's exact format string works perfectly without crashing!
df_clean = df_clean.withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:mm"))
df_clean = df_clean.withColumn("TotalPrice", col("Quantity") * col("UnitPrice"))

df_clean.show(5, truncate=False)
df_clean.printSchema()

df_clean = df_clean.withColumn("IsUK", when(col("Country") == "United Kingdom", lit(1)).otherwise(lit(0)))

df_clean.show(5, truncate=False)
display(df_clean)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- TotalPrice: double (nullable = true)
 |-- IsUK: integer (nullable = false)


In [ ]:
# 3. Filtering for high value transactions
high_value_df = df_clean.filter((col("TotalPrice") > 50) & (col("Quantity") > 10))
high_value_df.display()

In [ ]:
# 4. Grouping by items to get transaction metrics
product_stats_df = df_clean.groupBy("StockCode", "Description").agg(
    {"Quantity": "sum", "TotalPrice": "avg", "InvoiceNo": "count"}
).withColumnRenamed("sum(Quantity)", "total_quantity") \
 .withColumnRenamed("avg(TotalPrice)", "avg_price") \
 .withColumnRenamed("count(InvoiceNo)", "transaction_count")

product_stats_df.orderBy(col("total_quantity").desc()).display()

In [ ]:
# 5. Extract distinct customer mappings
customers_df = df_clean.select("CustomerID", "Country").distinct()
customers_df.display()

In [ ]:
# 6. Join the customer dataset
customer_transactions_df = df_clean.join(
    customers_df.select([i for i in customers_df.columns if i != 'Country']), 
    "CustomerID", 
    "inner"
)
customer_transactions_df.display()

In [ ]:
# 7. Aggregate spending patterns per customer
# Note: Legacy .cache() call removed here to comply with serverless execution
customer_summary_df = customer_transactions_df.groupBy("CustomerID", "Country").agg(
    {"TotalPrice": "sum", "InvoiceNo": "count"}
).withColumnRenamed("sum(TotalPrice)", "total_spent") \
 .withColumnRenamed("count(InvoiceNo)", "purchase_count")

customer_summary_df.orderBy(col("total_spent").desc()).display()

In [ ]:
# 8. Create temporary view for Spark SQL execution
df_clean.createOrReplaceTempView("retail")

In [ ]:
# 9. Simple bulk order filter query
basic_sql = spark.sql("""
    select CustomerID, Description, Quantity, UnitPrice
    from retail
    where Quantity > 80000
    order by Quantity desc
""")
basic_sql.display()

 CustomerID                         Description  Quantity  UnitPrice
      16446  WORLD WAR II TEMPLATE WESTIE REGLD     80995       2.08

In [ ]:
# 10. Map customer temporary view
df_clean.createOrReplaceTempView("customers")

In [ ]:
# 11. Country-level metric analysis query
# Fixed: Rectified alias spacing typo and matched column casing
country_summary_sql = spark.sql("""
    select c.Country, COUNT(distinct r.InvoiceNo) as invoice_count, round(sum(r.Quantity * r.UnitPrice), 2) as total_revenue
    from retail r
    join customers c ON r.CustomerID = c.CustomerID
    group by c.Country
    order by total_revenue desc
""")
country_summary_sql.display()

        Country  invoice_count  total_revenue
 United Kingdom          16649     7308391.47
    Netherlands             95      285446.34
           EIRE            260      265545.90
        Germany            457      228867.14
         France            389      209024.05
      Australia             57      138521.31
          Spain             90       61577.11
    Switzerland             51       56443.95
        Belgium             98       41196.30
         Sweden             36       38378.33

In [ ]:
# 12. Segment identification query
customes_pattern_sql = spark.sql("""
    select CustomerID, COUNT(distinct StockCode) as unique_products, AVG(Quantity) as avg_quantity
    from retail
    group by CustomerID
    having count(distinct StockCode) > 10
    order by unique_products desc
""")
customes_pattern_sql.display()

 CustomerID  unique_products  avg_quantity
      14911             1785     46.327610
      12748             1760      2.164319
      17841             1110      2.822295
      14606              801      2.222384
      14096              570      5.474635
      14298              516     49.208168
      15039              503      4.851086
      13089              497      6.143714
      13408              457     13.918903
      15311              443      8.474623

In [ ]:
# 13. Temporal revenue tracking query
# Fixed: Rectified misspelled group by column name and alias spacing typo
time_analysis_sql = spark.sql("""
    select YEAR(InvoiceDate) as year, MONTH(InvoiceDate) as month, SUM(TotalPrice) as monthly_revenue
    from retail
    group by YEAR(InvoiceDate), MONTH(InvoiceDate)
    order by year, month
""")
time_analysis_sql.display()

 year  month  monthly_revenue
 2010     12        572713.89
 2011      1        569750.02
 2011      2        447137.35
 2011      3        595500.76
 2011      4        469200.28
 2011      5        678512.44
 2011      6        661200.31
 2011      7        600090.87
 2011      8        645343.90
 2011      9        952838.22
 2011     10       1039318.40
 2011     11       1161816.20
 2011     12        342506.11